### Issue with byte-level tokenization
- While byte-level tokenization can alleviate the out-of-vocabulary issues faced by word-level tokenizers, tokenizing text into bytes results in extremely long input sequences. 
- This slows down model training, since a sentence with 10 words might only be 10 tokens long in a word-level language model, but could be 50 or more tokens long in a character-level model (depending on the length of the words). 
- Processing these longer sequences requires more computation at each step of the model. 
- Furthermore, language modeling on byte sequences is difficult because the longer input sequences create long-term dependencies in the data.

### Subword Tokenization
- **Subword tokenization** is a midpoint between word-level tokenizers and byte-level tokenizers. 
- Note that a byte-level tokenizer’s vocabulary has 256 entries (byte values are 0 to 255) because it can represent any char using upto 4 256 entries. 
- A subword tokenizer trades off a larger vocabulary size for better compression of the input byte sequence. 
    - For example, if the byte sequence b'the' often occurs in our raw text training data, assigning it an entry in the vocabulary would reduce this 3-token sequence to a single token.

#### How do we select these subword units to add to our vocabulary?
- propose to use byte-pair encoding (BPE), a compression algorithm that iteratively replaces (“merges”) the most frequent pair of bytes with a single, new unused index.
- this algorithm adds subword tokens to our vocabulary to maximize the compression of our input sequences—if a word occurs in our input text enough times, it’ll be represented as a single subword unit.
- Subword tokenizers with vocabularies constructed via BPE are often called **BPE tokenizers**.
- a byte-level BPE tokenizer, where the vocabulary items are bytes or merged sequences of bytes, which give us the best of both worlds in terms of out-of-vocabulary handling and manageable input sequence lengths. 
- The process of constructing the BPE tokenizer vocabulary is known as “training” the BPE tokenizer.

### BPE Tokenizer Training
The BPE tokenizer training procedure consists of three main steps.

#### Vocabulary initialization 
- The tokenizer vocabulary is a one-to-one mapping from bytestring token to integer ID. 
- Since we’re training a byte-level BPE tokenizer, our initial vocabulary is simply the set of all bytes. Since there are 256 possible byte values, our initial vocabulary is of size 256.

#### bpe_example

In [369]:
from collections import Counter
from collections import defaultdict

##### corpus

In [370]:
text = "low low low low low lower lower widest widest widest newest newest newest newest newest newest"

##### vocab
- We initialize our vocabulary with our special token <|endoftext|> and the 256 byte values.

In [371]:
vocab = {i: chr(i) for i in range(256)}  # Create a vocabulary mapping each byte value (0-255) to itself.

In [372]:
vocab[256] = "<|endoftext|>"
print(vocab)

{0: '\x00', 1: '\x01', 2: '\x02', 3: '\x03', 4: '\x04', 5: '\x05', 6: '\x06', 7: '\x07', 8: '\x08', 9: '\t', 10: '\n', 11: '\x0b', 12: '\x0c', 13: '\r', 14: '\x0e', 15: '\x0f', 16: '\x10', 17: '\x11', 18: '\x12', 19: '\x13', 20: '\x14', 21: '\x15', 22: '\x16', 23: '\x17', 24: '\x18', 25: '\x19', 26: '\x1a', 27: '\x1b', 28: '\x1c', 29: '\x1d', 30: '\x1e', 31: '\x1f', 32: ' ', 33: '!', 34: '"', 35: '#', 36: '$', 37: '%', 38: '&', 39: "'", 40: '(', 41: ')', 42: '*', 43: '+', 44: ',', 45: '-', 46: '.', 47: '/', 48: '0', 49: '1', 50: '2', 51: '3', 52: '4', 53: '5', 54: '6', 55: '7', 56: '8', 57: '9', 58: ':', 59: ';', 60: '<', 61: '=', 62: '>', 63: '?', 64: '@', 65: 'A', 66: 'B', 67: 'C', 68: 'D', 69: 'E', 70: 'F', 71: 'G', 72: 'H', 73: 'I', 74: 'J', 75: 'K', 76: 'L', 77: 'M', 78: 'N', 79: 'O', 80: 'P', 81: 'Q', 82: 'R', 83: 'S', 84: 'T', 85: 'U', 86: 'V', 87: 'W', 88: 'X', 89: 'Y', 90: 'Z', 91: '[', 92: '\\', 93: ']', 94: '^', 95: '_', 96: '`', 97: 'a', 98: 'b', 99: 'c', 100: 'd', 101: 'e'

##### Pre-tokenization
- For simplicity and to focus on the merge procedure, we assume in this example that pretokenization simply splits on whitespace.

In [373]:
corpus = text.strip().split()
print(corpus)

['low', 'low', 'low', 'low', 'low', 'lower', 'lower', 'widest', 'widest', 'widest', 'newest', 'newest', 'newest', 'newest', 'newest', 'newest']


In [374]:
count = Counter(corpus)
count

Counter({'newest': 6, 'low': 5, 'widest': 3, 'lower': 2})

- It is convenient to represent this as a dict[tuple[bytes, ...], int], e.g. {(l,o,w): 5, …}. 
- Note that even a single byte is a bytes object in Python. There is no byte type in Python to represent a single byte, just as there is no char type in Python to represent a single character.

In [375]:
def byte_tuple(k):
    return tuple(i.encode('utf-8') for i in k) # Convert each string in the tuple to its byte representation.

my_dict = { byte_tuple(k):v for k,v in count.items() if v > 1}
my_dict

{(b'l', b'o', b'w'): 5,
 (b'l', b'o', b'w', b'e', b'r'): 2,
 (b'w', b'i', b'd', b'e', b's', b't'): 3,
 (b'n', b'e', b'w', b'e', b's', b't'): 6}

##### Merges
- We first look at every successive pair of bytes and sum the frequency of the words where they appear

In [376]:

def common_pairs(my_dict):
    global vocab
    pairs = defaultdict(int)
    for k,v in my_dict.items():
        for i in range(len(k)-1):
            pairs[k[i]+k[i+1]]+=v
    pairs = sorted(pairs.items(), key=lambda x: (x[1], x[0]), reverse=True)
    vocab[len(vocab)] = pairs[0][0]
    return pairs[0][0]

- sort the pairs by their frequency in descending order and then by their byte value in descending order (lexicographically)

In [386]:
def merge_pair(pair, my_dict):
    new_dict = {}
    for k,v in my_dict.items():
        temp = []
        i = 0
        while i < len(k):
            if i < len(k) - 1 and k[i]+k[i+1] == pair:
                temp.append(k[i]+k[i+1])
                i += 2   # Skip the next character since it's part of the merged pair.
            else:
                temp.append(k[i])
                i += 1
        new_dict[tuple(temp)] = v
    return new_dict

for i in range(6):
    pair = common_pairs(my_dict)
    my_dict = merge_pair(pair, my_dict)
    print(my_dict)
    print(f"Iteration {i+1}: Merged pair {pair} into vocabulary. New vocabulary size: {len(vocab)}")
    
print(vocab)


{(b'low',): 5, (b'low', b'e', b'r'): 2, (b'w', b'i', b'd', b'est'): 3, (b'newest',): 6}
Iteration 1: Merged pair b'newest' into vocabulary. New vocabulary size: 264
{(b'low',): 5, (b'low', b'e', b'r'): 2, (b'wi', b'd', b'est'): 3, (b'newest',): 6}
Iteration 2: Merged pair b'wi' into vocabulary. New vocabulary size: 265
{(b'low',): 5, (b'low', b'e', b'r'): 2, (b'wid', b'est'): 3, (b'newest',): 6}
Iteration 3: Merged pair b'wid' into vocabulary. New vocabulary size: 266
{(b'low',): 5, (b'low', b'e', b'r'): 2, (b'widest',): 3, (b'newest',): 6}
Iteration 4: Merged pair b'widest' into vocabulary. New vocabulary size: 267
{(b'low',): 5, (b'lowe', b'r'): 2, (b'widest',): 3, (b'newest',): 6}
Iteration 5: Merged pair b'lowe' into vocabulary. New vocabulary size: 268
{(b'low',): 5, (b'lower',): 2, (b'widest',): 3, (b'newest',): 6}
Iteration 6: Merged pair b'lower' into vocabulary. New vocabulary size: 269
{0: '\x00', 1: '\x01', 2: '\x02', 3: '\x03', 4: '\x04', 5: '\x05', 6: '\x06', 7: '\x07', 8:

In [387]:
word = "lowest".encode("utf-8")   # work in bytes
tokenized_word = []
i = 0

while i < len(word):
    found = False

    for j in range(len(vocab) - 1, -1, -1):  # largest → smallest
        token = vocab[j]  # token is bytes
        if word[i:i+len(token)] == token:
            tokenized_word.append(vocab[j])
            i += len(token)
            found = True
            break

    if not found:
        raise ValueError(f"No matching token at position {i}")

print(tokenized_word)

[b'lowe', b'st']
